<div style="font-family: Arial, sans-serif; line-height: 1.6; color: #fab956; max-width: 1200px; margin: 0 auto; padding: 20px; border: 1px solid #000000; border-radius: 8px; background-color: #00183a74;">
  <h1 style="color: #c25719; border-bottom: 2px solid #c25719; padding-bottom: 8px; margin-top: 0;">
    Notebook Map
  </h1>
  
| # | Notebook | Content | CRISP-DM phase |
|---|---|---|---|
| 00 | `00_relatorio.ipynb` | Problem, data, literature, decisions | Business and data understanding |
| 01 | `lndb_exploration.ipynb` | LNDb exploration and the LNDb nodule table | Data understanding |
| 02 | `lidc_exploration.ipynb` | LIDC IDRI exploration and the LIDC nodule table | Data understanding |
| 03 | `03_tabela_nodulos.ipynb` | Joined table, study cohort, labels on both scales | Data preparation |
| 04 | `04_patches.ipynb` | 3D patches and their checks | Data preparation |
| 05 | `05_features.ipynb` | Radiologists' regions, PyRadiomics, robustness, feature filters | Data preparation |
| 06 | `06_modelo.ipynb` | Models, ways of pooling the datasets, nested validation, interpretability, fairness | Modelling and evaluation |

Shared code: `src/mass_detect/scripts/` (`config`, `schema`, `lidc`, `cohort`, `patches`, `qc`, `rois`, `radiomics_features`, `sensitivity`, `models`, `validation`). Unit tests: `pytest src/mass_detect/tests`.

  <h1 style="color: #c25719; border-bottom: 2px solid #c25719; padding-bottom: 8px; margin-top: 0;">
    Predicting Lung Nodule Malignancy from CT Data
  </h1>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   The problem and who uses it
  </h3>
  <p>
  The end user is a thoracic radiologist who reads a chest CT with lung nodules and must decide what to do with each one: nothing, a follow up scan, or further investigation (MacMahon et al., 2017). Our model gives a second opinion on the malignancy rating of a nodule that the radiologist has already found and outlined, on the original 1 to 5 scale and on a 1 to 3 scale that follows that clinical decision (benign, indeterminate, suspicious).
  </p>
  <p>
  Constraints we work under:
  </p>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;">The label is a subjective rating, and two radiologists agree only moderately, which sets a ceiling for any model.</li>
    <li style="margin-bottom: 8px;">The model must be explainable, because a radiologist will not act on a score they cannot question.</li>
  </ul>
  <p>
  This report uses training data only.
  </p>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   What data do we have?
  </h3>

  <p>
  
| Dataset | What it contains | How we use it |
|---|---|---|
| LNDb train | 236 CT scans, with 1219 unique findings (768 nodules), segmentation masks for nodules >= 3 mm | Main training data |
| LIDC IDRI | 1010 patients, 1018 CT series, up to four readings per nodule | Annotations through `pylidc`; patches cut from the DICOM files; pooled with LNDb in training and validation |
  </p>
  <p>
  Malignancy is evaluated in LNDb's file <strong>'chars_trainNodules.csv'</strong>, joined to the readings by row position, because RadID means different things in the two files.
  </p>
  
  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
    Which files to consider? And which to discard?
  </h3>

  <ol style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;">Kept:
      <ul style="padding-left: 20px; margin-bottom: 20px;">
      <li style="margin-bottom: 8px;">The merged findings file <strong>'trainNodules_gt.csv'</strong>. The unit of analysis is the nodule, not the reading, because one nodule can be read by up to three radiologists and counting it three times would leak information which is a crucial error we made sure not to make.</li>
      <li style="margin-bottom: 8px;"><strong>'chars_trainNodules.csv'</strong> for the malignancy ratings, <strong>'LNDbAcqParams.csv'</strong> for scanner information, and the masks for nodules of 3 mm or more.</li>
      </ul>
    </li>
    <li style="margin-bottom: 8px;">Dropped:
      <ul style="padding-left: 20px; margin-bottom: 20px;">
        <li style="margin-bottom: 8px;">The per patient diagnosis file of LIDC. It says whether a patient has cancer but not which nodule is the malignant one, so using it as a label would give every nodule of that patient the same wrong answer. Therefore, we did not consider it useful for our main goal.</li>
        <li style="margin-bottom: 8px;">Fleischner files and medical reports. Fleischner is a follow up recommendation for the patient, not a malignancy rating for each nodule, and the reports are free text, not a rating.</li>
        <li style="margin-bottom: 8px;">The original <strong>'.rar'</strong> archives, the X ray series of LIDC (DX and CR) and the macOS <strong>'._*'</strong> files.</li>
        </ul>
    </li>
    <li style="margin-bottom: 8px;">Practical Notes:</li>
      <ul style="padding-left: 20px; margin-bottom: 20px;">
        <li style="margin-bottom: 8px;"><strong>'allNods.csv'</strong> holds ratings already averaged, but its findings were merged differently (the same FindingID points to another place in 90 findings), so we rebuild the mean ourselves from the individual readings.</li>
      </ul>
    </li>
  </ol>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   Key Findings in the Literature
  </h3>

  <p>
  <strong>Size alone explains a lot:</strong> Baltatzis et al. (2021) show that a baseline that only uses nodule diameter reaches 83% to 85% accuracy on LIDC, and that the way labels are aggregated can matter more than the model. Mean and median aggregation give 1010 and 848 usable nodules respectively. We will always report a diameter only baseline and use it for every comparison. Our diameter only baseline uses the diameter measured on the radiologists' region.
  </p>
  <p>
  <strong>Splits must respect patients:</strong> Causey et al. (2018) list, among the common errors in evaluating nodule malignancy models, not keeping all the nodules of one patient on the same side of the split between training and validation. Nodules from one scan share scanner, protocol and anatomy, so splitting them leaks information (Kapoor and Narayanan, 2023), and we always validate by patient.
  </p>
  <p>
  <strong>Tools:</strong> PyRadiomics (van Griethuysen et al., 2017) is our feature extractor and the IBSI work (Zwanenburg et al., 2020) is the reference for feature definitions. The IBSI requires isotropic voxels for 3D texture features but does not prescribe a size, and its own reference configurations use 2 mm. We use 1 mm, the slice spacing of 289 of the 294 LNDb scans, with a fixed bin width of 25 HU; notebook 05 tests other voxel sizes with a sensitivity analysis.
  </p>
  <p>
  <strong>Domain shift:</strong> LNDb comes from one hospital with three Siemens scanner models, while LIDC is multi centre and multi vendor. LNDb was designed as an external set for models trained on LIDC. Both datasets enter training and validation. Notebook 06 compares ways of pooling them (plain pooling, a dataset indicator, harmonisation with ComBat (Orlhac et al., 2022) and aggregating several models).
  </p>
  <p>
  <strong>Pretrained features:</strong> The foundation model of Pai et al. (2024) only needs the lesion centroid. It is an option for future work, since our team decided not to use deep learning.
  </p>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   How the label is built
  </h3>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;"><strong>The unit is the nodule, not the reading.</strong> Counting readings would weigh a nodule by the number of its readers and could leak the same nodule across folds.</li>
    <li style="margin-bottom: 8px;"><strong>The mean, not the maximum.</strong> In LNDb, the maximum would raise the multi reader nodules rated 4 or more from 20 to 58. It would reward being read by more radiologists, and it would amplify reader bias: radiologist R3 averages 1.57 and R4 averages 2.72.</li>
    <li style="margin-bottom: 8px;"><strong>Rounded half up, floor(mean + 0.5).</strong> 1232 of the 3237 cohort nodules have a non integer mean, and 500 end exactly in .5. Python and numpy round .5 to the nearest even number, which would change 310 labels for no clinical reason; always rounding up (ceil) would change 334, and the median high of the readers 403. Rounding half up treats every tie between two readers in the more cautious direction.</li>
    <li style="margin-bottom: 8px;"><strong>The 1 to 3 scale comes from the rounded rating.</strong> Ratings 1 and 2 are benign, 3 is indeterminate, 4 and 5 are suspicious. Cutting the mean directly at 2.5 and 3.5 would give 396 nodules contradictory labels on the two scales; for example, 266 nodules rated 3 would become benign.</li>
    <li style="margin-bottom: 8px;"><strong>Class 3 is kept.</strong> It is 32% of the LNDb cohort and 47% of the LIDC IDRI cohort. Removing it makes the problem look easier than it is (Lei et al., 2020; Zhang et al., 2022; Baltatzis et al., 2021).</li>
    <li style="margin-bottom: 8px;"><strong>The label is noisy.</strong> In LNDb, 20% of the nodules read by several radiologists differ by 2 points or more; in LIDC IDRI 40% do, and 23% were called benign by one reader and suspicious by another.</li>
  </ul>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   Decisions
  </h3>

  | Decision | Choice | Why |
|----|----|----|
| Scope | Training data only | The test set has no labels |
| Unit of analysis | Unique nodule | Section above |
| Cohort | Nodule with a rating; 3 mm or more with a radiologist mask; LIDC groups of at most four annotations with a confirmed position. 622 LNDb + 2615 LIDC = 3237 nodules from 1062 patients | The same rule in both datasets |
| Label | floor(mean + 0.5) of the non zero ratings | Section above |
| Scales | 1 to 5 and 1 to 3 (derived from the rounded rating) | The task and the clinical decision |
| Region of interest | Consensus of the radiologists' masks (a voxel is kept when at least half of the readers drew it), plus each reader's mask | The pylidc rule; Ross et al. (2007); Carrazza et al. (2016) |
| Patch | 64 mm cube | 0 consensus regions cut at 64 mm, against 116 at 30 mm and 4 at 51 mm; 99.6% of the nodules keep a 9 mm margin (3σ of the widest LoG filter) |
| Voxel | Isotropic 1 mm | The IBSI asks for isotropic voxels without fixing their size; 1 mm is the slice spacing of 289 of the 294 LNDb scans; sensitivity analysis in notebook 05 |
| Features | PyRadiomics (original + LoG 1, 2 and 3 mm, bin width 25 HU, HU range); filters without the label: constant, missing, correlation 0.9, ICC ≥ 0.75 in both datasets; selection that uses the label only inside the cross validation | Demircioğlu (2021); Traverso et al. (2018); Koo and Li (2016) |
| Diameter | Measured on the consensus region | One definition for both datasets (Bankier et al., 2017) |
| Pooling LNDb and LIDC | One model trained and validated on both; pooling strategies compared in notebook 06 | Notebook 06 |
| Models | Diameter only baseline; ordinal logistic regression with an L2 penalty (McCullagh, 1980); random forest and others compared | Baltatzis et al. (2021) |
| Validation | Nested StratifiedGroupKFold by patient, repeated; intervals by resampling patients | Cawley and Talbot (2010); Varoquaux (2018); Field and Welsh (2007); Kapoor and Narayanan (2023) |
| Metrics | Primary, fixed in advance: the mean of the QWK in LNDb and the QWK in LIDC. Secondary: balanced accuracy, macro F1, MAE, log loss, calibration. Ceiling: agreement between two radiologists (1 to 5: LNDb .527, LIDC .466; 1 to 3: .439, .319) | Cohen (1968); He and Garcia (2009) |
| Interpretability and fairness | Permutation importance; subgroups by dataset, scanner and kernel, slice thickness, size and number of readers (LNDb has no sex or age) | Molnar (2022); required by the brief |
<p>
</p>
  <h1 style="color: #c25719; border-bottom: 2px solid #c25719; padding-bottom: 8px; margin-top: 0;">
    References
  </h1>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;">Ambroise, C. and McLachlan, G. J. (2002). Selection bias in gene extraction on the basis of microarray gene expression data. <i>PNAS</i>, 99(10), 6562 to 6566.</li>
    <li style="margin-bottom: 8px;">Armato, S. G. III et al. (2011). The Lung Image Database Consortium (LIDC) and Image Database Resource Initiative (IDRI). <i>Medical Physics</i>, 38(2), 915 to 931.</li>
    <li style="margin-bottom: 8px;">Baltatzis, V. et al. (2021). The pitfalls of sample selection: a case study on lung nodule classification. arXiv:2108.05386.</li>
    <li style="margin-bottom: 8px;">Bankier, A. A. et al. (2017). Recommendations for measuring pulmonary nodules at CT: a statement from the Fleischner Society. <i>Radiology</i>, 285(2), 584 to 600.</li>
    <li style="margin-bottom: 8px;">Carrazza, M., Kennedy, B., Rasin, A., Furst, J. and Raicu, D. (2016). Investigating the effects of majority voting on CAD systems: a LIDC case study. <i>Medical Imaging 2016: Computer Aided Diagnosis</i>, SPIE.</li>
    <li style="margin-bottom: 8px;">Causey, J. L., Zhang, J., Ma, S., Jiang, B., Qualls, J. A., Politte, D. G., Prior, F., Zhang, S., & Huang, X. (2018). <i>Highly accurate model for prediction of lung nodule malignancy with CT scans</i>. Scientific Reports, 8, 9286</li>
    <li style="margin-bottom: 8px;">Cawley, G. C. and Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. <i>JMLR</i>, 11, 2079 to 2107.</li>
    <li style="margin-bottom: 8px;"> Cohen, J. (1968). <i>Weighted kappa: Nominal scale agreement with provision for scaled disagreement or partial credit</i>. Psychological Bulletin, 70(4), 213 to 220. https://doi.org/10.1037/h0026256</li>
    <li style="margin-bottom: 8px;">Collins, G. S. et al. (2024). TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <i>BMJ</i>, 385, e078378.</li>
    <li style="margin-bottom: 8px;">Demircioğlu, A. (2021). Measuring the bias of incorrect application of feature selection when using cross validation in radiomics. <i>Insights into Imaging</i>, 12, 172.</li>
    <li style="margin-bottom: 8px;">Field, C. A. and Welsh, A. H. (2007). Bootstrapping clustered data. <i>Journal of the Royal Statistical Society B</i>, 69(3), 369 to 390.</li>
    <li style="margin-bottom: 8px;">Gneiting, T. (2011). Making and evaluating point forecasts. <i>Journal of the American Statistical Association</i>, 106(494), 746 to 762.</li>
    <li style="margin-bottom: 8px;"> He, H., & Garcia, E. A. (2009). <i>Learning from Imbalanced Data.</i> IEEE Transactions on Knowledge and Data Engineering, 21(9), 1263 to 1284. https://doi.org/10.1109/TKDE.2008.239</li>
    <li style="margin-bottom: 8px;">Johnson, W. E., Li, C. and Rabinovic, A. (2007). Adjusting batch effects in microarray expression data using empirical Bayes methods. <i>Biostatistics</i>, 8(1), 118 to 127.</li>
    <li style="margin-bottom: 8px;">Kapoor, S. and Narayanan, A. (2023). Leakage and the reproducibility crisis in machine-learning-based science. <i>Patterns</i>, 4(9), 100804.</li>
    <li style="margin-bottom: 8px;">Koo, T. K. and Li, M. Y. (2016). A guideline of selecting and reporting intraclass correlation coefficients for reliability research. <i>Journal of Chiropractic Medicine</i>, 15(2), 155 to 163.</li>
    <li style="margin-bottom: 8px;">Lambin, P. et al. (2017). Radiomics: the bridge between medical imaging and personalized medicine. <i>Nature Reviews Clinical Oncology</i>, 14, 749 to 762.</li>
    <li style="margin-bottom: 8px;"> Lei, Y., Zhu, H., Zhang, J., & Shan, H. (2020). <i>Meta Ordinal Regression Forest For Learning with Unsure Lung Nodules</i>. IEEE International Conference on Bioinformatics and Biomedicine (BIBM). https://arxiv.org/abs/2012.03480</li>
    <li style="margin-bottom: 8px;">MacMahon, H. et al. (2017). Guidelines for management of incidental pulmonary nodules detected on CT images: from the Fleischner Society 2017. <i>Radiology</i>, 284(1), 228 to 243.</li>
    <li style="margin-bottom: 8px;">McCullagh, P. (1980). Regression models for ordinal data. <i>Journal of the Royal Statistical Society B</i>, 42(2), 109 to 142.</li>
    <li style="margin-bottom: 8px;">Molnar, C. (2022). Interpretable Machine Learning (2nd ed.). christophm.github.io/interpretable-ml-book.</li>
    <li style="margin-bottom: 8px;">Mongan, J., Moy, L. and Kahn, C. E. (2020). Checklist for Artificial Intelligence in Medical Imaging (CLAIM). <i>Radiology: Artificial Intelligence</i>, 2(2), e200029.</li>
    <li style="margin-bottom: 8px;">Orlhac, F. et al. (2022). A guide to ComBat harmonization of imaging biomarkers in multicenter studies. <i>Journal of Nuclear Medicine</i>, 63(2), 172 to 179.</li>
    <li style="margin-bottom: 8px;">Pai, S. et al. (2024). Foundation model for cancer imaging biomarkers. <i>Nature Machine Intelligence</i>.</li>
    <li style="margin-bottom: 8px;">Pedrosa, J. et al. (2019). LNDb: a lung nodule database on computed tomography. arXiv:1911.08434.</li>
    <li style="margin-bottom: 8px;">PyRadiomics documentation, Customization (resampling and interpolators).</li>
    <li style="margin-bottom: 8px;">Raykar, V. C. et al. (2010). Learning from crowds. <i>Journal of Machine Learning Research</i>, 11, 1297 to 1322.</li>
    <li style="margin-bottom: 8px;">Regulation (EU) 2016/679 (GDPR); Regulation (EU) 2017/745 (MDR); Regulation (EU) 2024/1689 (AI Act).</li>
    <li style="margin-bottom: 8px;">Ross, J. C., Miller, J. V., Turner, W. D. and Kelliher, T. P. (2007). An analysis of early studies released by the Lung Imaging Database Consortium (LIDC). <i>Academic Radiology</i>, 14(11), 1382 to 1388.</li>
    <li style="margin-bottom: 8px;">Saerens, M., Latinne, P. and Decaestecker, C. (2002). Adjusting the outputs of a classifier to new a priori probabilities: a simple procedure. <i>Neural Computation</i>, 14(1), 21 to 41.</li>
    <li style="margin-bottom: 8px;">Shafiq-ul-Hassan, M. et al. (2017). Intrinsic dependencies of CT radiomic features on voxel size and number of gray levels. <i>Medical Physics</i>, 44(3), 1050 to 1062.</li>
    <li style="margin-bottom: 8px;">Shrout, P. E. and Fleiss, J. L. (1979). Intraclass correlations: uses in assessing rater reliability. <i>Psychological Bulletin</i>, 86(2), 420 to 428.</li>
    <li style="margin-bottom: 8px;">Tejani, A. S. et al. (2024). Checklist for Artificial Intelligence in Medical Imaging (CLAIM): 2024 update. <i>Radiology: Artificial Intelligence</i>.</li>
    <li style="margin-bottom: 8px;">Traverso, A. et al. (2018). Repeatability and reproducibility of radiomic features: a systematic review. <i>International Journal of Radiation Oncology, Biology, Physics</i>, 102(4), 1143 to 1158.</li>
    <li style="margin-bottom: 8px;">van Griethuysen, J. J. M. et al. (2017). Computational radiomics system to decode the radiographic phenotype. <i>Cancer Research</i>, 77(21), e104 to e107.</li>
    <li style="margin-bottom: 8px;">Varoquaux, G. (2018). Cross-validation failure: small sample sizes lead to large error bars. <i>NeuroImage</i>, 180, 68 to 77.</li>
    <li style="margin-bottom: 8px;">Wolpert, D. H. (1992). Stacked generalization. <i>Neural Networks</i>, 5(2), 241 to 259.</li>
    <li style="margin-bottom: 8px;"> Zhang, H., Gu, X., Zhang, M., Yu, W., Chen, L., Wang, Z., Yao, F., Gu, Y., & Yang, G.-Z. (2022). <i>Re-thinking and Re-labeling LIDC-IDRI for Robust Pulmonary Cancer Prediction</i>. https://arxiv.org/abs/2207.14238</li>
    <li style="margin-bottom: 8px;">Zhang, Y., Jenkins, D. F., Manimaran, S. and Johnson, W. E. (2018). Alternative empirical Bayes models for adjusting for batch effects in genomic studies. <i>BMC Bioinformatics</i>, 19, 262.</li>
    <li style="margin-bottom: 8px;">Zwanenburg, A. et al. (2019). Assessing robustness of radiomic features by image perturbation. <i>Scientific Reports</i>, 9, 614.</li>
    <li style="margin-bottom: 8px;">Zwanenburg, A. et al. (2020). The Image Biomarker Standardization Initiative: standardized quantitative radiomics for high-throughput image-based phenotyping. <i>Radiology</i>, 295(2), 328 to 338.</li>
  </ul>
</div>